# Full-resolution CUDA generator verification
Four independent batches of 64 fields at 256 × 256, plus one fixed-seed replay.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy.git"
REF = "4fcab71e8e253d39b6dd0728e472d2f719424882"   # output of `git rev-parse HEAD` for the commit to run
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = "/tmp/physicsnemo-fno-darcy"


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert not dirty, "the working tree is not clean"
print("commit", head)

Install the pinned PhysicsNeMo commit and the CPU audit versions of PyTorch, Warp, NumPy and SciPy into a fresh virtual environment. The next check reports the installed versions and confirms CUDA availability.

In [ ]:
import sys
VENV = "/tmp/darcy-verification-env"
subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)
PYTHON = f"{VENV}/bin/python"
subprocess.run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "-q", "pip"], check=True)
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
# Isolate from Kaggle's optional RAPIDS stack. Match the CPU audit's Warp and
# numerical libraries to make the cross-device comparison easier to interpret.
sh("python -m pip install -q -r requirements.txt torch==2.14.1 torchvision==0.29.1 "
   "warp-lang==1.18.0 numpy==2.5.3 scipy==1.18.1", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

## Generate, compare, and preserve evidence
References are sparse direct solves on the CPU.

In [ ]:
import json, shutil, zipfile
CHECK = f"{WORK}/cuda_batch64_validation"
from pathlib import Path
evidence = Path(CHECK)
evidence.mkdir(exist_ok=False)
output = evidence / "measurements"
(evidence / "nvidia-smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(evidence / "pip-freeze.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))
command = [PYTHON, "-u", "scripts/verify_cuda_batch64.py", "--out", str(output)]

with (evidence / "verification.log").open("w", buffering=1) as log:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, env={**os.environ, "MPLBACKEND": "Agg",
                                          "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = proc.wait()
report_path = output / "report.json"
report = json.loads(report_path.read_text()) if report_path.exists() else {}
complete = status == 0 and report.get("status") == "complete"
(evidence / "notebook_run.json").write_text(json.dumps(
    {"exit_status": status, "elapsed_seconds": time.time() - T0, "complete": complete}, indent=2
) + "\n")
with zipfile.ZipFile(f"{WORK}/cuda_batch64_validation.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(evidence.rglob("*")):
        if path.is_file() and (not complete or path.suffix != ".npz"):
            archive.write(path, path.relative_to(evidence))
if complete:
    shutil.rmtree(evidence)
else:
    print("Verification incomplete. The archive contains available evidence and fields for CPU analysis.")
print("Archive:", f"{WORK}/cuda_batch64_validation.zip")
